In [ ]:
import requests
import json
from pathlib import Path
import pandas as pd
import config
import re

## Cleaning response from APIs

_First practice how you would have cleaned if the api didn't have a parameter to do so automatically...but since remote job has only a little data set, we will be relying on adzuna, jooble, and arbeitnow._

In [ ]:
def load_file(file_path: str) -> list[dict]:
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_file(file_path: str, data: list[dict]) -> None:
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [ ]:
def fix_mojibake(obj):
    if isinstance(obj, str):
        try:
            return obj.encode('latin-1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return obj
    elif isinstance(obj, dict):
        return {fix_mojibake(k): fix_mojibake(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [fix_mojibake(item) for item in obj]
    return obj

def fetch_from_remoteok(filter):
    url = "https://remoteok.com/api"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    response.encoding = 'utf-8'
    raw = response.json()
    return fix_mojibake(raw)

_first load & clean from remoteOK_

data format should be:
```json
{
  "title": "Machine Learning Engineer",
  "company": "Example Inc.",
  "location": "Worldwide",
  "salary": "$120k-$160k",
  "description": "...",
  "posted_date": "...",
  "work_type": "remote"
}
```

In [ ]:
def determine_work_type(job):
    text = (
        job.get("slug", "") + " " +
        job.get("position", "") + " " +
        job.get("description", "") + " " +
        job.get("location", "")
    ).lower()

    remote = "remote" in text
    hybrid = "hybrid" in text
    onsite = "on-site" in text or "onsite" in text
    if remote:
        return "remote"

    if onsite:
        return "onsite"

    if hybrid or (remote and onsite):
        return "hybrid"

    return "unknown"

In [ ]:
# file paths
# raw file, the first created
raw_file_path = Path.cwd() / ".." / "data" / "raw" / "remoteok.json"
raw_file_path.parent.mkdir(parents=True, exist_ok=True)

# cleaned file
cleaned_file_path = Path.cwd() / ".." / "data" / "cleaned" / "remoteok.json"
cleaned_file_path.parent.mkdir(parents=True, exist_ok=True)

# matched file, matched and scored one
matched_file_path = Path.cwd() / ".." / "data" / "matched" / "remoteok.json"
matched_file_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_data = fetch_from_remoteok()
print(len(raw_data))

cleaned_data = list()

save_file(raw_file_path, raw_data)

for i in range(1, len(raw_data)):
    # collect what's needed
    job_position = raw_data[i].get("position", "unknown")
    hiring_company = raw_data[i].get("company", "unknown")
    job_location = raw_data[i].get("location", "unknown")
    
    min_salary = raw_data[i].get("salary_min", 0)
    max_salary = raw_data[i].get("salary_max", 0)
    
    job_description = raw_data[i].get("description")
    posted_date = raw_data[i].get("date")
    work_type = determine_work_type(raw_data[i])
    job_tags = raw_data[i].get("tags", [])
    
    # append the data
    cleaned_data.append(
        {
            "job_name": job_position,
            "company": hiring_company,
            "location": job_location,
            "min_salary": min_salary,
            "max_salary": max_salary,
            "description": job_description,
            "posted_date": posted_date,
            "work_type": work_type,
            "tags": job_tags,
        }
    )

print(len(cleaned_data))
# print(json.dumps(cleaned_data, indent=4))

save_file(cleaned_file_path, cleaned_data)
print(f"File at {cleaned_file_path} created successfully!")


In [ ]:
data = load_file(cleaned_file_path)

target_roles = [
    "data scientist",
    "machine learning",
    "ml engineer",
    "data analyst",
    "ai engineer",
    "data engineer"
]

results = list()
data_copy = data.copy()

for job in data_copy:
    position = job["job_name"].lower()
    description = job["description"].lower()
    tags = " ".join(job["tags"]).lower()

    job['matches'] = {
                "job_name": [],
                "description": [],
                "tags": []
                }
    
    for role in target_roles:
        if role in position:
            print(f"Found a match in job name: {job['job_name']} → {role}")
            
            job['matches']['job_name'].append(role)
            
            if job not in results:
                results.append(job)
        if role in description:
            print(f"Found a match in job description: {job['job_name']} → {role}")
            
            job['matches']['description'].append(role)
            
            if job not in results:
                results.append(job)
        if role in tags:
            print(f"Found a match in job tags: {job['job_name']} → {role}")
            
            job['matches']['tags'].append(role)
            
            if job not in results:
                results.append(job)

for job in results:
    job['score'] = 0
    
    if job['matches']['job_name']:
        job['score'] += 3
    if job['matches']['tags']:
        job['score'] += 2
    if job['matches']['description']:
        job['score'] += 1

# print(json.dumps(results, indent=4))

save_file(matched_file_path, results)
print(f"Successfully saved the matching datas to {matched_file_path}")


In [ ]:
name_matches = 0
description_matches = 0
tag_matches = 0

for job in results:
    if job["matches"]["job_name"]:
        name_matches += 1

    if job["matches"]["description"]:
        description_matches += 1

    if job["matches"]["tags"]:
        tag_matches += 1

print(f"Title matches: {name_matches}")
print(f"Description matches: {description_matches}")
print(f"Tag matches: {tag_matches}")

temp = results.copy()
for job in temp:
    if job["score"] >= 5:
        print(f"High score job: {job['job_name']} with score {job['score']}")
    if job["score"] < 3:
        print(f"Low score job: {job['job_name']} with score {job['score']}")
        results.remove(job)
        
save_file(matched_file_path, results)


# Now the real work
Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

Curabitur pretium tincidunt lacus. Nulla gravida orci a odio. Nullam varius, turpis et commodo pharetra, est eros bibendum elit, nec luctus magna felis sollicitudin mauris. Integer in mauris eu nibh euismod gravida. Duis ac tellus et risus vulputate vehicula. Donec lobortis risus a elit. Etiam sit amet orci eget eros faucibus tincidunt. Duis leo. Sed fringilla mauris sit amet nibh. Donec sodales sagittis magna. Sed consequat, leo eget bibendum sodales, augue velit cursus nunc, quis gravida magna mi a libero. Fusce vulputate eleifend sapien. Vestibulum purus quam, scelerisque ut, mollis sed, nonummy id, metus. Nullam accumsan lorem in dui. Cras ultricies mi eu turpis hendrerit fringilla. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia Curae; In ac dui quis mi consectetuer lacinia. Nam pretium turpis et arcu. Duis arcu.

# Phase 1: Data collection

In [ ]:
target_roles = [
    "data scientist",
    "machine learning",
    "ml engineer",
    "data analyst",
    "ai engineer",
    "data engineer"
]

In [ ]:
def determine_work_type(job: dict):
    text = " ".join([
        job.get("name", ""),
        job.get("title", ""),
        job.get("description", ""),
        job.get("contract_type", ""),
        job.get("contract_time", ""),
        job.get("location", {}).get("display_name", ""),
        " ".join(job.get("location", {}).get("area", []))
    ]).lower()

    hybrid_keywords = [
        "hybrid",
        "part remote",
        "partially remote"
    ]

    remote_keywords = [
        "remote",
        "work from home",
        "work-from-home",
        "wfh",
        "work from anywhere",
        "fully distributed",
        "distributed team",
        "telecommute",
        "home-based"
    ]

    onsite_keywords = [
        "on-site",
        "onsite",
        "on site",
        "office-based",
        "office based"
    ]

    if any(keyword in text for keyword in hybrid_keywords):
        return "hybrid"

    if any(keyword in text for keyword in remote_keywords):
        return "remote"

    if any(keyword in text for keyword in onsite_keywords):
        return "onsite"

    return "unknown"

In [ ]:
def load_file(file_path: str) -> list[dict]:
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_file(file_path: str, data: list[dict], mode: str = 'w') -> None:
    with open(file_path, mode, encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [ ]:
def clean_matched(*matches: dict) -> dict:
    dict_keys = []
    result_dict = {}
    for match in matches:
        for key in match.keys():
            if key not in dict_keys:
                dict_keys.append(key)
                result_dict[key] = [match[key]]
            else:
                result_dict[key].append(match[key])
        
    return result_dict

In [ ]:
def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    if isinstance(obj, (list, set, tuple)):
        return tuple(make_hashable(item) for item in obj)
    return obj

### Source 1: Adzuna
we now load 1,800 jobs from adzuna that we can work with *(6 roles \* 6 countries \* 10 pages \* 5 results/page)*

In [ ]:
adzuna_app_id = config.ADZUNA_APP_ID
adzuna_app_key = config.ADZUNA_APP_KEY

ADZUNA_RESULTS_PER_PAGE = 5

adzuna_countries = [
    "us",
    "gb",
    "ca",
    "au",
    "de",
    "in"
]

def fetch_jobs_from_adzuna(role, page, country, results):
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"

    params = {
        "app_id": adzuna_app_id,
        "app_key": adzuna_app_key,
        "what": role,
        "results_per_page": results
    }

    response = requests.get(url, params=params)

    response.raise_for_status()    

    data = response.json()

    return data["results"]

adzuna_raw_data = []
for role in target_roles:
    for country in adzuna_countries:
        for page in range(1, 21, 2):
            jobs = fetch_jobs_from_adzuna(role, page, country, ADZUNA_RESULTS_PER_PAGE)

            adzuna_raw_data.extend(jobs)

raw_adzuna_path = Path.cwd() / ".." / "data" / "raw" / "adzuna.json"
raw_adzuna_path.parent.mkdir(parents=True, exist_ok=True)

save_file(raw_adzuna_path, adzuna_raw_data)

print(f"File with {len(adzuna_raw_data)} entries saved to {raw_adzuna_path}")

*then we clean the raw data and extract the data we wanna work with*

In [ ]:
adzuna_data = load_file(raw_adzuna_path)

adzuna_cleaned_data = list()

for job in adzuna_data:
    # print(json.dumps(job["location"]["area"], indent=4))
    adzuna_cleaned_data.append(
        {
        "job_name": job.get("title", ""),
        "company": job.get("company", {}).get("display_name", ""),
        "country": job.get("location", {}).get("area", [])[0] if job.get("location", {}).get("area") else "",
        "location": job.get("location", {}).get("display_name", ""),
        "min_salary": job.get("salary_min", 0),
        "max_salary": job.get("salary_max", 0),
        "description": job.get("description", "").replace("\n", " ").strip(),
        "posted_date": job.get("created", ""),
        "work_type": determine_work_type(job),
        "tags": "; ".join([job.get("category", {}).get("tag", "")]),
        "source": "adzuna"
        }
    )

cleaned_adzuna_path = Path.cwd() / ".." / "data" / "cleaned" / "adzuna.csv"
cleaned_adzuna_path.parent.mkdir(parents=True, exist_ok=True)

adzuna_df = pd.DataFrame(adzuna_cleaned_data)
adzuna_df = adzuna_df.drop_duplicates().reset_index(drop=True)

adzuna_df.to_csv(cleaned_adzuna_path)
print(f"File with {len(adzuna_df)} entries saved to {cleaned_adzuna_path}")

### Source 2: Arbeit

load 1150 *((20 / 2) * 115)* entries from Arbeit by using `time.sleep` to avoid `tooManyRequests` error..and used paging to catch as much data as possible.

In [ ]:
import time

def fetch_from_arbeit(page):
    response = requests.get(f"https://arbeitnow.com/api/job-board-api?page={page}")

    response.raise_for_status()

    data = response.json()

    return data["data"]

raw_arbeit_data = []
for page in range(0, 20, 2):
    raw: list[dict] = fetch_from_arbeit(page)
    raw_arbeit_data.extend(raw)
    
    # to avoid hitting the arbeit rate limit, we can add a delay between requests
    time.sleep(60)  # sleep for 1 minute
    
print("File length:", len(raw_arbeit_data))
print(json.dumps(raw_arbeit_data, indent=4))


In [ ]:
print(len(raw_arbeit_data))
print(json.dumps(raw_arbeit_data[0], indent=4))

def filter_jobs(job: dict) -> bool:
    slug = job.get("slug", "").replace("-", " ")
    title = job.get("title", "")
    description = job.get("description", "")
    tags = " ".join(job.get("tags", []))

    job['score'] = 0
    job['matched'] = []

    for role in target_roles:
        if role in title:
            job['score'] += 5
            job['matched'].append({'job_title': role})
        if role in slug:
            job['score'] += 3
            job['matched'].append({'slug': role})
        if role in tags:
            job['score'] += 2
            job['matched'].append({'tags': role})
        if role in description:
            job['score'] += 1
            job['matched'].append({'description': role})

    return job.get("score", 0) > 0

raw_arbeit_final = []

for data in raw_arbeit_data:
    if filter_jobs(data):
        raw_arbeit_final.append(data)

raw_arbeit_path = Path.cwd() / ".." / "data" / "raw" / "arbeit.json"
raw_arbeit_path.parent.mkdir(parents=True, exist_ok=True)

save_file(raw_arbeit_path, raw_arbeit_final)
print(f"File with {len(raw_arbeit_final)} entries saved to {raw_arbeit_path}")

now clean and save it

In [ ]:
# clean country data for arbeitnow.com
import re

# Comprehensive lookup for common cities/regions in your data
LOCATION_MAPPING = {
    # Germany
    "germany": "Germany", "ger": "Germany", "gb": "United Kingdom", "de": "Germany",
    "stuttgart": "Germany", "munich": "Germany", "münchen": "Germany", 
    "cologne": "Germany", "köln": "Germany", "berlin": "Germany", 
    "leipzig": "Germany", "hürth": "Germany", "bielefeld": "Germany", 
    "chemnitz": "Germany", "dresden": "Germany", "bonn": "Germany", 
    "frankfurt": "Germany", "darmstadt": "Germany", "düsseldorf": "Germany",
    "hamburg": "Germany", "sachsen": "Germany", "brandenburg": "Germany",
    "liveeo": "Germany",
    
    # United Kingdom
    "uk": "United Kingdom", "united kingdom": "United Kingdom", "england": "United Kingdom",
    "london": "United Kingdom", "manchester": "United Kingdom", 
    "bicester": "United Kingdom", "cardiff": "United Kingdom", "cambridge": "United Kingdom",
    
    # United States
    "united states": "United States", "california": "United States", "mountain view": "United States"
}

def extract_country(location_str):
    if not isinstance(location_str, str) or not location_str.strip():
        return None  # Handles empty/blank entries (like index 2, 10, 54, 65, 66)
    
    text = location_str.lower()
    
    # 1. Multi-country string check (e.g., index 8)
    if "or" in text or "," in text:
        matched_countries = []
        for term, country in LOCATION_MAPPING.items():
            if re.search(r'\b' + re.escape(term) + r'\b', text):
                if country not in matched_countries:
                    matched_countries.append(country)
        
        # Check specific listed countries in index 8
        for country in ["Nigeria", "Ethiopia", "India", "Rwanda"]:
            if country.lower() in text and country not in matched_countries:
                matched_countries.append(country)
                
        if len(matched_countries) > 1:
            return ", ".join(matched_countries)
        elif len(matched_countries) == 1:
            return matched_countries[0]

    # 2. Single direct match check
    for term, country in LOCATION_MAPPING.items():
        if re.search(r'\b' + re.escape(term) + r'\b', text):
            return country

    # 3. Handle pure Remote/All Offices without location context
    if "remote" in text or "all offices" in text:
        return "Remote"

    return "Unknown"

In [ ]:
arbeit_data = load_file(raw_arbeit_path)

arbeit_cleaned_data = list()

for job in arbeit_data:
    arbeit_cleaned_data.append(
        {
        "job_name": job.get("title", ""),
        "company": job.get("company_name", ""),
        "country": extract_country(job.get("location", "")), # to-do
        "location": job.get("location", ""),
        "min_salary": job.get("salary_min", 0),
        "max_salary": job.get("salary_max", 0),
        "description": job.get("description", "").replace("\n", " ").strip(),
        "posted_date": job.get("created_at", ""),
        "work_type": "remote" if job.get("remote") else "onsite",
        "tags": "; ".join(job.get("tags", [])),
        "score": job.get("score", 0),
        "matched": make_hashable(clean_matched(*job.get("matched", [{}]))), # to-do
        "source": "arbeit"
        }
    )

cleaned_arbeit_path = Path.cwd() / ".." / "data" / "cleaned" / "arbeit.csv"
cleaned_arbeit_path.parent.mkdir(parents=True, exist_ok=True)

arbeit_df = pd.DataFrame(arbeit_cleaned_data)

# Convert string to list of countries
arbeit_df = arbeit_df.drop_duplicates().reset_index(drop=True)
arbeit_df.to_csv(cleaned_arbeit_path)

print(f"File with {len(arbeit_df)} entries saved to {cleaned_arbeit_path}")

### Source 3: Muse

In [ ]:
muse_locations = [
    "United States",
    "Canada",
    "United Kingdom",
    "Germany",
    "France",
    "Netherlands",
    "Australia",
    "India"
]

def fetch_from_muse(page, location):
    url = "https://www.themuse.com/api/public/jobs"

    params = {
        "page": page,
        "location": location
    }

    response = requests.get(url, params=params)

    response.raise_for_status()

    data = response.json()

    return data['results']

muse_raw_data = list()
for location in muse_locations:
    for page in range(100):
        data = fetch_from_muse(page, location)
        muse_raw_data.extend(data)

In [ ]:
print(len(muse_raw_data))
# print(json.dumps(muse_raw_data, indent=4))

def filter_categories(data: dict):
    contents = data['contents']
    name = data['name']
    short_name = data['short_name'].replace("-", " ")
    category = data.get("categories", [])[0].get("name", "") if data["categories"] else ''

    data['score'] = 0
    data['matched'] = []
    matched = data['matched']

    for role in target_roles:
        if role in name:
            data['score'] += 5
            matched.append({'job_title': role})
        if role in short_name:
            data['score'] += 3
            matched.append({'slug': role})
        if role in category:
            data['score'] += 2
            matched.append({'tags': role})
        if role in contents:
            data['score'] += 1
            matched.append({'description': role})

    return data['score'] > 0

final_raw_muse = list()
for data in muse_raw_data:
    if filter_categories(data):
        final_raw_muse.append(data)

print(len(final_raw_muse))
print(json.dumps(final_raw_muse, indent=4))

raw_muse_path = Path.cwd() / ".." / "data" / "raw" / "muse.json"
raw_muse_path.parent.mkdir(parents=True, exist_ok=True)

save_file(raw_muse_path, final_raw_muse)

In [ ]:
def get_exact_location(data: list[dict]) -> tuple:
    """Returns `work_type`, `country`, and `location` **respectively** given an input like:  
        ```
        data: list[dict] = [  
            {  
                "name": "Flexible / Remote"  
            },  
            {  
                "name": "New York, NY"  
            }  
        ],
        ```
    """
    
    if not data:
        return "unknown", "N/A", "N/A"
    
    work_type, country, location = "unknown", "N/A", "N/A"
    
    temp = data.copy()
    for component in temp:
        if '/' in component['name']:
            data.remove(component)
            work_type = determine_work_type(component)
        elif ',' in component['name']:
            location, country = component['name'].split(",")
            
            if country.strip() in ["CA", "NY", "DC"]:
                country = "United States"
        

    return work_type, country.strip(), location.strip()


In [ ]:
raw_muse_path = Path.cwd() / ".." / "data" / "raw" / "muse.json"
# finally, clean and save
muse_data = load_file(raw_muse_path)

# print("".join([json.dumps(muse['locations'], indent=4) for muse in muse_data]))

muse_cleaned_data = list()

for job in muse_data:
    work_type, country, location = get_exact_location(job.get("locations", {}))
    
    muse_cleaned_data.append(
        {
        "job_name": job.get("name", ""),
        "company": job.get("company", {}).get("name", ""),
        "country": country,
        "location": location,
        "min_salary": job.get("min_salary", 0),
        "max_salary": job.get("max_salary", 0),
        "description": job.get("contents", "").replace("\n", " ").strip(),
        "posted_date": job.get("publication_date", ""),
        "work_type": work_type,
        "score": job.get("score", 0),
        "matched": make_hashable(clean_matched(*job.get("matched", [{}]))),
        "source": "muse"
        }
    )

cleaned_muse_path = Path.cwd() / ".." / "data" / "cleaned" / "muse.csv"
cleaned_muse_path.parent.mkdir(parents=True, exist_ok=True)

# print(len(muse_cleaned_data))
# print(json.dumps(muse_cleaned_data, indent=4))
muse_df = pd.DataFrame(muse_cleaned_data)
muse_df = muse_df.drop_duplicates().reset_index(drop=True)

muse_df.to_csv(cleaned_muse_path)
print(f"Successfully saved the cleaned version with {len(muse_df)} entries to {cleaned_muse_path}")


## Working on:
```
def extract_years_of_experience(description: str) -> int:
    # working on this in Jupyter notebook
    pass

def extract_required_edu_background(description: str) -> str:
    # working on this in my Jupyter notebook
    pass

def extract_required_tools(description: str) -> list[str]:
    # working on this in my jupyter notebook
    pass
```

In [ ]:
with open("data/raw/arbeit.json", 'r', encoding='utf-8') as file:
    raw_arbeit = json.load(file)

def extract_required_edu_background(data: dict) -> tuple[str, ...]:
    bachelors_keywords: list[str] = [
        "bachelor",
        "bachelor's",
        "bsc",
        "b.s.",
        "bs"
    ]
    
    doctorate_keywords: list[str] = [
        "phd",
        "ph.d",
        "doctorate"
    ]
    
    masters_keywords: list[str] = [
        "master's",
        "msc",
        "m.s.",
        "master degree",
        "masters in "
    ]
    
    desc_lower: str = data["description"].lower()
    return_values: list = []
    
    if any(key in desc_lower for key in doctorate_keywords):
        return_values.append("PhD")
    if any(key in desc_lower for key in masters_keywords):
        return_values.append("Master's")
    if any(key in desc_lower for key in bachelors_keywords):
        return_values.append("Bachelor's")
    
    return tuple(return_values)

for job in raw_arbeit:
    edu_back = extract_required_edu_background(job)
    job['edu_back'] = edu_back if edu_back else "Not found"

# print("\n".join(edu for job in raw_arbeit for edu in job['edu_back'] if edu != "Not found"))

print(tuple([]))

counter = 0
c = 0
for job in raw_arbeit:
    if job['edu_back'] != "Not found":
        counter += 1
    else:
        c += 1
    print(job['edu_back'])
    
print(counter, c)

In [38]:
sample_texts = [
    # 1. Basic: proven track record on
    "Experience: 5 years proven track record on machine learning systems",

    # 2. Basic: proven experience on
    "Experience: 5 years proven experience on data engineering projects",

    # 3. Case variation
    "Experience: 5 years Proven Track Record On cybersecurity systems",

    # 4. Multiple spaces
    "Experience: 5 years proven   track   record   on cloud infrastructure",

    # 5. Multiple spaces with experience
    "Experience: 5 years proven   experience   on artificial intelligence projects",

    # 6. With other linking words around it
    "Experience: 5 years in Python, proven track record on backend systems",

    # 7. Longer realistic job description
    "Qualifications: 7+ years leading engineering teams and proven track record on delivering large-scale distributed systems",

    # 8. HTML-style tag before it
    "Experience[H]: 6 years proven experience on machine learning platforms",

    # 9. SHOULD NOT MATCH — missing "on"
    "Experience: 5 years proven track record in machine learning systems",

    # 10. SHOULD NOT MATCH — wrong phrase
    "Experience: 5 years proven history on machine learning systems",
]

In [48]:
import pandas as pd
import numpy as np

arbeit = pd.read_csv("data/cleaned/arbeit.csv")
pd.set_option("display.max_colwidth", None)

sample_texts = np.array([])

for i in range(40, 50):
    sample_texts = np.append(sample_texts, arbeit.loc[i, "description"])

muse = pd.read_csv("data/cleaned/muse.csv")
pd.set_option("display.max_colwidth", None)

for i in range(40, 50):
    sample_texts = np.append(sample_texts, muse.loc[i, "description"])


print(sample_texts.shape)

sample_texts = sample_texts.tolist()
print(sample_texts)

pd.Series(sample_texts).to_clipboard(index=False)

(20,)
['[NEWLINE]Your Mission[P]This is a company built for growth, not comfort. [NEWLINE][NEWLINE]When you join exmox, you\'re stepping onto a global, highly competitive playing field, solving real problems in environments that push you beyond your comfort zone.[NEWLINE][NEWLINE]We build high-performing consumer products in one of the most competitive industries there is: mobile gaming, leading one of its largest innovations, a rewarded user engagement and acquisition platform that helps publishers acquire and retain players through rewarding experiences.[NEWLINE] [NEWLINE] Senior Product Manager (f/m/x) \xa0isn\'t just about shipping features, it\'s about building products that perform. Product at exmox means understanding the bigger business picture, considering cross-functional impact across Engineering, Data, Marketing, and Operations, and turning strategy into product decisions that create real value, knowing every change you ship affects millions of users. At exmox, we\'re also 

In [39]:
import re, json

space = r"\s{0,2}?"
html_tags = r"(?:\[NEWLINE\]|\[BULLET\]|\[P\]|\[LIST\]|\[H\])"
linking_words = r"(?i:in|with|building|leading|working\s+(?i:in|with|on)|shipping|of|as|across|within|demonstrated\s+experience|proven\s+(?:track\s+record|experience)\s+on|leveraging|integrating|developing|implementing)"
looking_for_pattern = r"We\sare\slooking(?:\sfor|\sfor\ssomeone|\sfor\ssomeone\swho\shas)?:?"

with_pattern = rf"(?:with\s+at\s+least|at\s+least|with{space}(?:~|(?:about|approximately|a\sminimum\sof))|a\sminimum\sof){space}"
same_sentence_wildcard = rf"(?:(?!{html_tags})[^.\n;]){{0,50}}?"
range_separator = rf"{space}(?:-|to){space}"

textual = r"(?:one|two|three|four|five|six|seven|eight|nine|ten)"
numeric = r"(?:1[0-5]|[1-9])"
years = r"\b(?:years|year\(s\)|year)(?!\w)"

possible_numbers = rf"(?:{numeric}|{textual})"
captured_number = rf"(?P<years>{possible_numbers}(?:{range_separator}{possible_numbers})?{space}\+?)"
unnamed_captured_number = rf"(?:{possible_numbers}(?:{range_separator}{possible_numbers})?{space}\+?)"

main_pattern = rf"{captured_number}{space}{years}"
unnamed_main_pattern = rf"{unnamed_captured_number}{space}{years}"
linked_information_pattern = rf"(?:(?!{with_pattern}{unnamed_captured_number}|{unnamed_main_pattern}|{html_tags})[^.\n;])+"

field_info_after = rf"(?P<field>{linking_words}\s+{linked_information_pattern})"
field_info_middle = rf"(?P<field>(?:(?!{unnamed_main_pattern}|{html_tags})[^.\n;]){{0,50}}?)"

info_after_brackets = rf"(?P<field>{linked_information_pattern})"
experience_captured_after = rf"(?P<field>{same_sentence_wildcard}experience\s+{linking_words}\s+{linked_information_pattern})"

case_1 = rf"(?i:(?:{with_pattern}|with\s+){main_pattern}{info_after_brackets})"

case_2 = rf"\({main_pattern}:?\)[:\s]*{info_after_brackets}"
case_3 = rf"(?:Experience)(?:\[H\])?:?{same_sentence_wildcard}{main_pattern}\s+{field_info_after}"

case_4 = rf"(?i:{main_pattern}{experience_captured_after})"
case_5 = rf"(?i:{main_pattern}{field_info_middle}experience)"

case_6 = rf"(?:Qualifications?)(?:\[H\])?:?{same_sentence_wildcard}{main_pattern}\s+{field_info_after}"
case_7 = rf"(?:Qualifications?)(?:\[H\])?:?{field_info_middle}{main_pattern}"

case_8 = rf"{textual}{space}\((?P<years>{numeric}{space}\+?)\){space}{years}{same_sentence_wildcard}experience\s+{field_info_after}"
case_9 = rf"{textual}{space}\((?P<years>{numeric}{space}\+?)\){space}{years}{field_info_middle}experience"

case_10 = rf"{looking_for_pattern}{same_sentence_wildcard}{main_pattern}\s+{field_info_after}"
case_11 = rf"{looking_for_pattern}{field_info_middle}{main_pattern}"

patterns = [case_1, case_2, case_3, case_4, case_5, case_6, case_7, case_8, case_9, case_10, case_11]

success = 0
iter_count = 0
captured_json = {}

for text in sample_texts:
    remaining = text
    iter_count += 1
    pattern_count = 0
    for pattern in patterns:
        pattern_count += 1
        i = 1
        while match := re.search(pattern, remaining):
            success += 1
            if iteration_found := captured_json.get(iter_count-1):
                if pattern_found := iteration_found.get(pattern_count):
                    iteration_found.update({f"{pattern_count}({i})": match.groupdict()})
                else:
                    iteration_found.update({pattern_count: match.groupdict()})
            else:
                captured_json.update({iter_count-1: {pattern_count: match.groupdict()}})
            remaining = remaining[:match.start()] + remaining[match.end():]

print(json.dumps(captured_json, indent=4))

{
    "0": {
        "3": {
            "years": "5",
            "field": "proven track record on machine learning systems"
        }
    },
    "1": {
        "3": {
            "years": "5",
            "field": "proven experience on data engineering projects"
        }
    },
    "2": {
        "3": {
            "years": "5",
            "field": "Proven Track Record On cybersecurity systems"
        }
    },
    "3": {
        "3": {
            "years": "5",
            "field": "proven   track   record   on cloud infrastructure"
        }
    },
    "4": {
        "3": {
            "years": "5",
            "field": "proven   experience   on artificial intelligence projects"
        }
    },
    "5": {
        "3": {
            "years": "5",
            "field": "in Python, proven track record on backend systems"
        }
    },
    "6": {
        "6": {
            "years": "7+",
            "field": "leading engineering teams and proven track record on delivering large-sca

In [ ]:
# extract_years_of_experience_testor

    import pandas as pd
    import numpy as np
    import sys

    arbeit = pd.read_csv("data/cleaned/arbeit.csv")
    pd.set_option("display.max_colwidth", None)

    sample_texts = np.array([])

    for i in range(40, 50):
        sample_texts = np.append(sample_texts, arbeit.loc[i, "description"])

    muse = pd.read_csv("data/cleaned/muse.csv")
    pd.set_option("display.max_colwidth", None)

    for i in range(40, 50):
        sample_texts = np.append(sample_texts, muse.loc[i, "description"])


    print(sample_texts.shape)

    sample_texts = sample_texts.tolist()
    print(extract_years_of_experience(sample_texts[int(sys.argv[1])]))

In [2]:
import json
j = {}
for i in range(20):
    j[i]=[""]
    
with open("debugv-2.json", "w") as file:
    json.dump(j, file, indent=4)